# Imports

In [ ]:
import sqlite3
from datetime import datetime
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageTk
import tkinter as tk
from tkinter import ttk, messagebox
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

BASE_DIR = Path.cwd()
DB_PATH = BASE_DIR / "trashcam.db"
WASTE_CANDIDATES = ["recycle", "compost", "landfill"]

# Load Models

In [ ]:
image_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224",
)

waste_image_router = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-large-patch14",
)

text_embedder = SentenceTransformer("all-mpnet-base-v2")

CATEGORY_TEXTS = {
    "recycle": (
        "recyclable materials such as clean paper cardboard aluminum cans glass bottles and rigid plastic containers"
    ),
    "compost": (
        "compostable organic waste such as fruit peels vegetables food scraps coffee grounds tea bags and yard trimmings"
    ),
    "landfill": (
        "landfill trash such as contaminated mixed waste wrappers foam ceramics diapers and non recyclable debris"
    ),
}

WASTE_IMAGE_LABELS = [
    "a photo of recyclable waste like paper cardboard plastic bottle aluminum can or glass bottle",
    "a photo of compostable food and organic waste like banana peel fruit vegetable scraps",
    "a photo of landfill trash like chip bags styrofoam dirty mixed garbage",
]
IMAGE_LABEL_TO_CATEGORY = {
    WASTE_IMAGE_LABELS[0]: "recycle",
    WASTE_IMAGE_LABELS[1]: "compost",
    WASTE_IMAGE_LABELS[2]: "landfill",
}
category_names = list(CATEGORY_TEXTS.keys())
category_embeddings = text_embedder.encode(
    [CATEGORY_TEXTS[name] for name in category_names],
    convert_to_tensor=True,
    normalize_embeddings=True,
)

def detect_item(frame_bgr: np.ndarray):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    item_preds = image_classifier(pil_img, top_k=5)
    top_item = item_preds[0]["label"]
    top_conf = float(item_preds[0]["score"])
    return top_item, top_conf, item_preds


def classify_disposal_from_image(frame_bgr: np.ndarray):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)
    preds = waste_image_router(pil_img, candidate_labels=WASTE_IMAGE_LABELS)
    top = preds[0]
    disposal = IMAGE_LABEL_TO_CATEGORY[top["label"]]
    disposal_conf = float(top["score"])
    return disposal, disposal_conf, preds


def classify_disposal_from_text(item_label: str):
    q = text_embedder.encode(item_label, convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(q, category_embeddings)[0].detach().cpu().numpy()
    best_idx = int(np.argmax(sims))
    disposal = category_names[best_idx]
    disposal_conf = float(sims[best_idx])
    return disposal, disposal_conf


print("Pretrained models ready")

# Create SQLite DB

In [ ]:
def init_db(db_path: Path = DB_PATH):
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS trash_events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event_time TEXT NOT NULL,
                item_label TEXT NOT NULL,
                disposal_type TEXT NOT NULL,
                confidence REAL NOT NULL,
                image_path TEXT
            )
            """
        )
        conn.commit()


def save_event(item_label: str, disposal_type: str, confidence: float, image_path: str | None = None, db_path: Path = DB_PATH):
    now = datetime.now().isoformat(timespec="seconds")
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            """
            INSERT INTO trash_events (event_time, item_label, disposal_type, confidence, image_path)
            VALUES (?, ?, ?, ?, ?)
            """,
            (now, item_label, disposal_type, float(confidence), image_path),
        )
        conn.commit()


def report_counts(start_date: str, end_date: str, db_path: Path = DB_PATH):
    start_dt = datetime.fromisoformat(start_date + "T00:00:00")
    end_dt = datetime.fromisoformat(end_date + "T23:59:59")
    with sqlite3.connect(db_path) as conn:
        cur = conn.execute(
            """
            SELECT disposal_type, COUNT(*)
            FROM trash_events
            WHERE event_time BETWEEN ? AND ?
            GROUP BY disposal_type
            """,
            (start_dt.isoformat(timespec="seconds"), end_dt.isoformat(timespec="seconds")),
        )
        rows = cur.fetchall()
    counts = {"recycle": 0, "compost": 0, "landfill": 0}
    for disposal_type, total in rows:
        counts[disposal_type] = total
    return counts


init_db()
print("Database ready at", DB_PATH)

# Tkinter App

In [ ]:
class TrashCamApp:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("TrashCam Classifier")
        self.root.geometry("1100x700")
        self.cap = cv2.VideoCapture(0)
        if not self.cap.isOpened():
            raise RuntimeError("Could not open camera. Check webcam permissions.")
        self.current_frame = None
        self.latest_item_label = "unknown"
        self.latest_item_conf = 0.0
        self.stable_item_label = "unknown"
        self.stable_item_conf = 0.0
        self.label_history = []
        self.max_history = 6
        self.last_live_infer_at = datetime.min
        self.live_infer_interval_seconds = 0.8
        self.pending_record = None
        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True)
        self.classifier_tab = ttk.Frame(self.notebook)
        self.report_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.classifier_tab, text="Classifier")
        self.notebook.add(self.report_tab, text="Reports")
        self._build_classifier_tab()
        self._build_report_tab()
        self._update_preview()
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)

    def _build_classifier_tab(self):
        top = ttk.Frame(self.classifier_tab, padding=(12, 10, 12, 4))
        top.pack(fill="x")
        ttk.Label(
            top,
            text="Position the item so it fills most of the green box, then click Classify.",
            font=("Segoe UI", 12, "bold"),
        ).pack(anchor="w")
        body = ttk.Frame(self.classifier_tab, padding=12)
        body.pack(fill="both", expand=True)
        left = ttk.Frame(body)
        left.pack(side="left", fill="both", expand=True)
        right = ttk.Frame(body)
        right.pack(side="right", fill="y", padx=(12, 0))
        self.preview_label = ttk.Label(left)
        self.preview_label.pack(fill="both", expand=True)
        ttk.Label(right, text="Live Detection", font=("Segoe UI", 11, "bold")).pack(anchor="w", pady=(0, 4))
        self.live_item_var = tk.StringVar(value="Current item: analyzing...")
        ttk.Label(right, textvariable=self.live_item_var, wraplength=320, justify="left").pack(anchor="w", pady=(0, 8))
        self.stable_item_var = tk.StringVar(value="Stable item: waiting...")
        ttk.Label(right, textvariable=self.stable_item_var, wraplength=320, justify="left").pack(anchor="w", pady=(0, 12))
        ttk.Label(right, text="Manual item (optional)").pack(anchor="w")
        self.manual_item_entry = ttk.Entry(right, width=36)
        self.manual_item_entry.pack(anchor="w", pady=(4, 10))
        ttk.Button(right, text="Classify", command=self.classify_current_item).pack(fill="x", pady=(0, 8))
        action_row = ttk.Frame(right)
        action_row.pack(fill="x", pady=(0, 10))
        ttk.Button(action_row, text="Redo", command=self.redo_classification).pack(side="left", fill="x", expand=True, padx=(0, 4))
        ttk.Button(action_row, text="Next (Save)", command=self.next_and_save).pack(side="left", fill="x", expand=True, padx=(4, 0))
        self.result_var = tk.StringVar(value="No classification yet.")
        ttk.Label(right, text="Classification Result", font=("Segoe UI", 11, "bold")).pack(anchor="w", pady=(4, 4))
        ttk.Label(right, textvariable=self.result_var, wraplength=320, justify="left").pack(anchor="w")
        ttk.Separator(right, orient="horizontal").pack(fill="x", pady=10)
        ttk.Button(right, text="Refresh Report Totals", command=self.run_report).pack(fill="x")

    def _build_report_tab(self):
        frame = ttk.Frame(self.report_tab, padding=12)
        frame.pack(fill="both", expand=True)
        ttk.Label(frame, text="Report Date Range", font=("Segoe UI", 12, "bold")).grid(row=0, column=0, columnspan=2, sticky="w", pady=(0, 10))
        ttk.Label(frame, text="Start date (YYYY-MM-DD)").grid(row=1, column=0, sticky="w")
        self.start_entry = ttk.Entry(frame, width=20)
        self.start_entry.grid(row=1, column=1, sticky="w", padx=(10, 0), pady=4)
        ttk.Label(frame, text="End date (YYYY-MM-DD)").grid(row=2, column=0, sticky="w")
        self.end_entry = ttk.Entry(frame, width=20)
        self.end_entry.grid(row=2, column=1, sticky="w", padx=(10, 0), pady=4)
        today = datetime.now().strftime("%Y-%m-%d")
        self.start_entry.insert(0, today)
        self.end_entry.insert(0, today)
        ttk.Button(frame, text="Run Report", command=self.run_report).grid(row=3, column=0, columnspan=2, sticky="w", pady=(10, 10))
        self.report_text = tk.Text(frame, width=60, height=10)
        self.report_text.grid(row=4, column=0, columnspan=2, sticky="nsew")
        frame.grid_columnconfigure(0, weight=1)
        frame.grid_rowconfigure(4, weight=1)

    def _roi_bounds(self, frame: np.ndarray):
        h, w = frame.shape[:2]
        box_w = int(w * 0.55)
        box_h = int(h * 0.55)
        x1 = (w - box_w) // 2
        y1 = (h - box_h) // 2
        x2 = x1 + box_w
        y2 = y1 + box_h
        return x1, y1, x2, y2

    def _draw_overlay(self, frame: np.ndarray):
        draw = frame.copy()
        x1, y1, x2, y2 = self._roi_bounds(frame)
        cv2.rectangle(draw, (x1, y1), (x2, y2), (0, 220, 0), 3)
        cv2.putText(
            draw,
            "Fill this green box with your item",
            (x1, max(25, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 220, 0),
            2,
            cv2.LINE_AA,
        )
        return draw

    def _extract_roi(self, frame: np.ndarray):
        x1, y1, x2, y2 = self._roi_bounds(frame)
        return frame[y1:y2, x1:x2].copy()

    def _should_run_live_inference(self):
        delta = datetime.now() - self.last_live_infer_at
        return delta.total_seconds() >= self.live_infer_interval_seconds

    def _update_live_item(self, roi_frame: np.ndarray):
        if not self._should_run_live_inference():
            return
        self.last_live_infer_at = datetime.now()
        try:
            item_label, item_conf, _ = detect_item(roi_frame)
            self.latest_item_label = item_label
            self.latest_item_conf = item_conf
            self.live_item_var.set(f"Current item: {item_label} ({item_conf:.2f})")
            self.label_history.append((item_label, item_conf))
            if len(self.label_history) > self.max_history:
                self.label_history.pop(0)
            if len(self.label_history) >= 4:
                recent_labels = [x[0] for x in self.label_history[-4:]]
                if len(set(recent_labels)) == 1:
                    self.stable_item_label = recent_labels[0]
                    recent_confs = [x[1] for x in self.label_history[-4:]]
                    self.stable_item_conf = float(sum(recent_confs) / len(recent_confs))
                    self.stable_item_var.set(
                        f"Stable item: {self.stable_item_label} ({self.stable_item_conf:.2f})"
                    )
        except Exception:
            pass

    def _update_preview(self):
        ok, frame = self.cap.read()
        if ok:
            self.current_frame = frame.copy()
            roi = self._extract_roi(frame)
            self._update_live_item(roi)
            draw = self._draw_overlay(frame)
            rgb = cv2.cvtColor(draw, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb)
            pil_img = pil_img.resize((720, 480))
            tk_img = ImageTk.PhotoImage(pil_img)
            self.preview_label.configure(image=tk_img)
            self.preview_label.image = tk_img
        self.root.after(30, self._update_preview)

    def _selected_item_label(self):
        manual = self.manual_item_entry.get().strip()
        if manual:
            return manual, 1.0, "manual"
        if self.stable_item_label != "unknown":
            return self.stable_item_label, self.stable_item_conf, "stable"
        if self.latest_item_label != "unknown":
            return self.latest_item_label, self.latest_item_conf, "live"
        return None, 0.0, "none"

    def classify_current_item(self):
        if self.current_frame is None:
            messagebox.showwarning("No frame", "No camera frame available yet.")
            return
        item_label, item_conf, source = self._selected_item_label()
        if item_label is None:
            messagebox.showwarning("No item", "No item detected yet. Center item in green box or type it manually.")
            return
        roi = self._extract_roi(self.current_frame)
        try:
            if source == "manual":
                disposal, disposal_conf = classify_disposal_from_text(item_label)
                category_source = "text-embedding"
            else:
                disposal, disposal_conf, _ = classify_disposal_from_image(roi)
                category_source = "image-clip"
        except Exception as ex:
            messagebox.showerror("Classification error", str(ex))
            return
        self.pending_record = {
            "item_label": item_label,
            "item_conf": float(item_conf),
            "disposal": disposal,
            "disposal_conf": float(disposal_conf),
            "source": source,
            "category_source": category_source,
        }
        self.result_var.set(
            f"Item: {item_label} ({source})\n"
            f"Item confidence: {item_conf:.2f}\n"
            f"Category: {disposal}\n"
            f"Category confidence: {disposal_conf:.2f}\n"
            f"Category model: {category_source}\n"
            "Click Next (Save) to log this result, or Redo to discard."
        )

    def redo_classification(self):
        self.pending_record = None
        self.result_var.set("Classification cleared. Reposition item and click Classify.")

    def next_and_save(self):
        if not self.pending_record:
            messagebox.showinfo("Nothing to save", "Run Classify first, then click Next (Save).")
            return
        save_event(
            self.pending_record["item_label"],
            self.pending_record["disposal"],
            self.pending_record["item_conf"],
            None,
        )
        self.result_var.set(
            f"Saved\n"
            f"Item: {self.pending_record['item_label']}\n"
            f"Category: {self.pending_record['disposal']}"
        )
        self.pending_record = None
        self.manual_item_entry.delete(0, "end")

    def run_report(self):
        start_date = self.start_entry.get().strip()
        end_date = self.end_entry.get().strip()
        try:
            counts = report_counts(start_date, end_date)
        except ValueError:
            messagebox.showerror("Invalid date", "Use YYYY-MM-DD format for both dates.")
            return
        except Exception as ex:
            messagebox.showerror("Report error", str(ex))
            return
        lines = [
            f"Range: {start_date} to {end_date}",
            "",
            f"Recycle:  {counts['recycle']}\n",
            f"Compost:  {counts['compost']}\n",
            f"Landfill: {counts['landfill']}\n",
        ]
        self.report_text.delete("1.0", "end")
        self.report_text.insert("1.0", "\n".join(lines))

    def on_close(self):
        if self.cap is not None and self.cap.isOpened():
            self.cap.release()
        self.root.destroy()

# Run

In [ ]:
root = tk.Tk()
app = TrashCamApp(root)
root.mainloop()